## QUEST 2

In [11]:
import pandas as pd 
import sqlite3

## 1) create a connection to the database using the library sqlite3

In [12]:
try:

    connection = sqlite3.connect('../data/checking-logs.sqlite')
    cursor = connection.cursor()
except Exception:
    print("The connection is failed :(")

## 2) create a new table datamart in the database by joining the tables pageviews and checker using only one query

* the table should have the following columns: uid, labname, first_commit_ts,
first_view_ts
* first_commit_ts is just a new name of the column timestamp from the checker
table, it shows the first commit from a particular lab and from a particular
user
* first_view_ts is the first visit of a user to the table pageviews, timestamp
when a user visited the newsfeed
* status = ’ready’ should still be a filter
* numTrials = 1 should still be a filter
* labnames should still be from the list: ’laba04’, ’laba04s’, ’laba05’, ’laba06’,
’laba06s’, ’project1’
* the table should contain only the users (uids with user_*) and not the admins
* first_commit_ts and first_view_ts should be parsed as datetime64[ns]

In [13]:

query = """ 
select c.uid, c.labname, datetime(c.timestamp) as first_commit_ts, datetime(p.datetime) as first_view_ts from checker c
left join pageviews p on p.uid = c.uid 
where c.status = "ready" and 
	  c.numTrials = 1 and 
	  c.labname in ('laba04', 'laba04s', 'laba05', 'laba06','laba06s', 'project1') and 
	  c.uid like 'user_%' and 
	  (p.datetime = (select min(p2.datetime) from pageviews p2
	  						where p2.uid = c.uid) or p.datetime is null) and
	  c.timestamp in (select min(c2.timestamp) from checker c2
	  						where c2.uid = c.uid and c2.labname = c.labname)
"""
# pd.io.sql.read_sql(query, connection) создает таблицу sql-запросом, но выдает TypeError: 'NoneType' object is not iterable
# лучше использовать pd.io.sql.execute(), если без парсинга и c create table datamart as

datamart = pd.io.sql.read_sql_query(query, connection, parse_dates=['first_commit_ts', 'first_view_ts'])

In [14]:
display(datamart)

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02,NaT
1,user_4,laba04,2020-04-17 11:33:17,NaT
2,user_4,laba04s,2020-04-17 11:48:41,NaT
3,user_17,project1,2020-04-18 07:56:45,2020-04-18 10:56:55
4,user_30,laba04,2020-04-18 13:36:53,2020-04-17 22:46:26
...,...,...,...,...
135,user_23,laba06,2020-05-21 08:34:10,NaT
136,user_19,laba06s,2020-05-21 13:27:06,2020-04-21 20:30:38
137,user_23,laba06s,2020-05-21 14:29:15,NaT
138,user_17,laba06,2020-05-21 15:21:31,2020-04-18 10:56:55


Создалась таблица datamart:

## 3) using Pandas methods, create two dataframes: test and control

* test should have the users that have the values in first_view_ts
* control should have the users that have missing values in first_view_ts
* replace the missing values in the control with the average first_view_ts of the test users, we will use this value for the future analysis
* save both tables into the database, you will use them in the next exercises

In [15]:
test = datamart[datamart["first_view_ts"].notna()]
display(test.head())

,uid,labname,first_commit_ts,first_view_ts
3,user_17,project1,2020-04-18 07:56:45,2020-04-18 10:56:55
4,user_30,laba04,2020-04-18 13:36:53,2020-04-17 22:46:26
7,user_30,laba04s,2020-04-18 14:51:37,2020-04-17 22:46:26
8,user_14,laba04,2020-04-18 15:14:00,2020-04-18 10:53:52
11,user_14,laba04s,2020-04-18 22:30:30,2020-04-18 10:53:52


In [16]:
print(test.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 59 entries, 3 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.3+ KB
None


In [17]:
mean = test["first_view_ts"].mean()

In [18]:
control = datamart[datamart["first_view_ts"].isna()]
control = control.fillna({"first_view_ts":mean})
display(control)

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02,2020-04-27 00:40:05.322034176
1,user_4,laba04,2020-04-17 11:33:17,2020-04-27 00:40:05.322034176
2,user_4,laba04s,2020-04-17 11:48:41,2020-04-27 00:40:05.322034176
5,user_2,laba04,2020-04-18 13:42:35,2020-04-27 00:40:05.322034176
6,user_2,laba04s,2020-04-18 13:51:22,2020-04-27 00:40:05.322034176
...,...,...,...,...
126,user_2,laba06s,2020-05-19 14:45:03,2020-04-27 00:40:05.322034176
132,user_6,laba06s,2020-05-20 14:50:07,2020-04-27 00:40:05.322034176
134,user_7,laba06s,2020-05-20 23:05:37,2020-04-27 00:40:05.322034176
135,user_23,laba06,2020-05-21 08:34:10,2020-04-27 00:40:05.322034176


In [19]:
control.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 81 entries, 0 to 137
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    81 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


In [20]:
try:
    test.to_sql("test", connection)
    control.to_sql("control", connection)
except ValueError:
    print("Этот файл запускается не в первый раз, а значит, эти таблицы уже существуют в СУБД :)")

Этот файл запускается не в первый раз, а значит, эти таблицы уже существуют в СУБД :)


In [21]:
connection.close()